# Computer Vision --- Lecture 6 Hands-On
## Vision Transformers


Today you write attention. Not call it -- write it, and its backward pass, and then
assemble a Vision Transformer out of the pieces and train it.

Every number on today's slides was produced by this notebook. You can check all of them.

| Section | Topic |
|---|---|
| 0 | Setup |
| 1 | Scaled dot-product attention |
| 2 | Why $\sqrt{d_k}$ -- measured |
| 3 | Multi-head attention, gradient-checked |
| 4 | Patch embedding is a strided convolution |
| 5 | Permutation equivariance, and why position must be added |
| 6 | A complete transformer block |
| 7 | Training a real ViT, and reading its attention |
| 8 | The data-efficiency crossover |
| 9 | The same thing in PyTorch |

Cells marked **TODO** are yours. Exercises at the end.

---
## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

np.set_printoptions(precision=4, suppress=True, linewidth=120)
plt.rcParams["figure.figsize"] = (10, 3.2)
rng = np.random.default_rng(0)
ACCENT, RED, GREEN, ORANGE = "#1F4E79", "#B03A2E", "#1E8449", "#E67E22"

In [ ]:
def softmax(s, axis=-1):
    """Numerically stable softmax -- subtract the max before exponentiating."""
    s = s - s.max(axis=axis, keepdims=True)
    e = np.exp(s)
    return e / e.sum(axis=axis, keepdims=True)


def rel_error(a, b):
    return np.max(np.abs(a - b) / np.maximum(1e-12, np.abs(a) + np.abs(b)))


def numeric_grad(f, x, h=1e-5):
    """Centred finite differences over every entry of x."""
    grad = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=["multi_index"], op_flags=["readwrite"])
    while not it.finished:
        i = it.multi_index
        old = x[i]
        x[i] = old + h; fp = f()
        x[i] = old - h; fm = f()
        x[i] = old
        grad[i] = (fp - fm) / (2 * h)
        it.iternext()
    return grad

---
## 1. Scaled Dot-Product Attention

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V$$

Three lines of NumPy. Write it, then check the properties that make it attention rather
than an arbitrary matrix product.

In [ ]:
def attention(Q, K, V):
    """Q: (n_q, d_k)  K: (n_k, d_k)  V: (n_k, d_v)  ->  (n_q, d_v), plus the weights."""
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    A = softmax(scores, axis=-1)
    return A @ V, A


# the slide's soft-lookup example, reproduced
K = np.array([[1.0, 0.0], [0.0, 1.0], [0.7, 0.7], [-1.0, 0.2]])
V = np.array([[10.0], [20.0], [30.0], [40.0]])
q = np.array([[0.9, 0.5]])

out, A = attention(q * np.sqrt(2), K, V)     # undo the 1/sqrt(d) for this toy
print("attention weights:", A[0].round(3))
print("rows sum to      :", A.sum(1))
print("hard lookup would return:", V[(K @ q[0]).argmax(), 0])
print("attention returns       :", out[0, 0].round(2))

Two properties worth stating out loud:

1. **Every row of `A` sums to 1.** The output is a convex combination of the values --
   attention never invents magnitude, it only redistributes.
2. **`A` is not a parameter.** It was computed from the input. Change the input, get a
   different `A`. That is the entire difference from a convolution kernel.

In [ ]:
# a sanity check on claim 1, at a realistic size
r = np.random.default_rng(0)
Q_, K_, V_ = r.standard_normal((8, 16)), r.standard_normal((12, 16)), r.standard_normal((12, 5))
out_, A_ = attention(Q_, K_, V_)
print("A shape", A_.shape, " output shape", out_.shape)
print("max deviation of row sums from 1:", np.abs(A_.sum(1) - 1).max())
print("output is inside the convex hull of V:",
      bool((out_.min() >= V_.min() - 1e-9) and (out_.max() <= V_.max() + 1e-9)))

**TODO 1.** Attention with all-equal scores returns the mean of the values. Verify it:
build `Q` and `K` so that every score is identical (the easiest way is `Q = 0`), and confirm
the output equals `V.mean(0)` for every query. Then perturb one key slightly and watch how
far the output moves -- this is the sense in which attention is a *soft* lookup.

In [ ]:
# TODO 1 -- your code here

---
## 2. Why $\sqrt{d_k}$

If $q$ and $k$ have independent zero-mean unit-variance entries, then
$q\cdot k = \sum_{i=1}^{d_k} q_ik_i$ is a sum of $d_k$ independent terms, so

$$\mathrm{Var}(q\cdot k) = d_k \qquad\Longrightarrow\qquad \text{std} = \sqrt{d_k}$$

**This is exactly Lecture 5's variance argument** -- the same algebra that gave us
$\mathrm{Var}(w) = 1/n$. Here the consequence is different: large logits push softmax into
saturation, where it becomes $\arg\max$ and its gradient dies.

In [ ]:
r = np.random.default_rng(1)
print(f"{'d_k':>6} {'std(q.k)':>10} {'sqrt(d)':>9} | "
      f"{'max p (raw)':>12} {'max p (scaled)':>15} | {'grad raw':>10} {'grad scaled':>12}")
for d in (4, 16, 64, 256, 512):
    q = r.standard_normal((4000, d))
    Kb = r.standard_normal((4000, 8, d))
    s = np.einsum("nd,nkd->nk", q, Kb)
    p_raw, p_sc = softmax(s), softmax(s / np.sqrt(d))
    # sum_j p_j(1-p_j) measures how much gradient the softmax lets through
    g_raw = (p_raw * (1 - p_raw)).sum(1).mean()
    g_sc = (p_sc * (1 - p_sc)).sum(1).mean()
    print(f"{d:6d} {s.std():10.2f} {np.sqrt(d):9.2f} | "
          f"{p_raw.max(1).mean():12.3f} {p_sc.max(1).mean():15.3f} | "
          f"{g_raw:10.2e} {g_sc:12.3f}")

Read the last two columns. Unscaled, the softmax at $d_k = 512$ puts 96% of its weight on a
single key and passes back a gradient twelve times smaller than the scaled version. The
network would be unable to learn to attend anywhere else.

With the $1/\sqrt{d_k}$ division, every column is flat in $d_k$ -- which is the point. The
scaling is not a tuning constant; it is the unique factor that makes the layer's behaviour
independent of head width.

**TODO 2.** The scaling assumes unit-variance $q$ and $k$. Break the assumption: multiply `q`
by 10 and re-run the table. Does $1/\sqrt{d_k}$ still save you? Relate what you find to the
role of LayerNorm immediately before the QKV projection in the block you will build in
section 6.

In [ ]:
# TODO 2 -- your code here

---
## 3. Multi-Head Attention, With Its Backward Pass

$$\mathrm{MSA}(X) = [\mathrm{head}_1; \dots; \mathrm{head}_h]\,W_O,
\qquad \mathrm{head}_i = \mathrm{Attention}(XW_Q^i, XW_K^i, XW_V^i)$$

In practice you do not loop over heads. One `Wqkv` of shape $(D, 3D)$ produces all of
$Q, K, V$ at once; reshaping splits them into heads.

In [ ]:
def msa_forward(X, Wqkv, Wo, h):
    """X: (B, n, D).  Returns (B, n, D) and a cache for the backward pass."""
    B, n, D = X.shape
    dh = D // h
    qkv = X @ Wqkv                                  # (B, n, 3D)
    q, k, v = np.split(qkv, 3, axis=-1)
    rs = lambda t: t.reshape(B, n, h, dh).transpose(0, 2, 1, 3)
    q, k, v = rs(q), rs(k), rs(v)                   # (B, h, n, dh)
    scores = q @ k.transpose(0, 1, 3, 2) / np.sqrt(dh)
    A = softmax(scores, axis=-1)
    ctx = (A @ v).transpose(0, 2, 1, 3).reshape(B, n, D)
    return ctx @ Wo, (X, q, k, v, A, ctx, Wqkv, Wo, h)


def msa_backward(dout, cache):
    X, q, k, v, A, ctx, Wqkv, Wo, h = cache
    B, n, D = X.shape
    dh = D // h
    dWo = ctx.reshape(-1, D).T @ dout.reshape(-1, D)
    dctx = (dout @ Wo.T).reshape(B, n, h, dh).transpose(0, 2, 1, 3)
    dA = dctx @ v.transpose(0, 1, 3, 2)
    dv = A.transpose(0, 1, 3, 2) @ dctx
    # softmax Jacobian, applied row-wise: dS = A * (dA - sum_j dA_j A_j)
    dscores = A * (dA - (dA * A).sum(-1, keepdims=True)) / np.sqrt(dh)
    dq = dscores @ k
    dk = dscores.transpose(0, 1, 3, 2) @ q
    un = lambda t: t.transpose(0, 2, 1, 3).reshape(B, n, D)
    dqkv = np.concatenate([un(dq), un(dk), un(dv)], axis=-1)
    dWqkv = X.reshape(-1, D).T @ dqkv.reshape(-1, 3 * D)
    dX = dqkv @ Wqkv.T
    return dX, dWqkv, dWo

In [ ]:
# gradient check
r = np.random.default_rng(0)
B, n, D, h = 3, 7, 12, 3
X = r.standard_normal((B, n, D))
Wqkv = r.standard_normal((D, 3 * D)) * 0.2
Wo = r.standard_normal((D, D)) * 0.2
G = r.standard_normal((B, n, D))        # an arbitrary upstream gradient

out, cache = msa_forward(X, Wqkv, Wo, h)
dX, dWqkv, dWo = msa_backward(G, cache)

obj = lambda: (msa_forward(X, Wqkv, Wo, h)[0] * G).sum()
for name, arr, ana in [("X", X, dX), ("Wqkv", Wqkv, dWqkv), ("Wo", Wo, dWo)]:
    ng = numeric_grad(obj, arr)
    print(f"  d{name:5s} rel error {rel_error(ng, ana):.2e}   "
          f"max abs diff {np.abs(ng - ana).max():.2e}")

### A gradient that is exactly zero

Add a bias to the queries and keys, `qkv = X @ Wqkv + b`, and gradient-check `b`. The
query part behaves normally. The **key part comes out exactly zero**, and not by accident:

$$\text{score}_{ij} = \frac{q_i\cdot(k_j + c)}{\sqrt{d_k}} = \frac{q_i\cdot k_j}{\sqrt{d_k}} + \frac{q_i\cdot c}{\sqrt{d_k}}$$

The added term is the same for every $j$, and softmax is invariant to a constant shift
along the axis it normalises. So the key bias cannot affect the output at all.

This is worth knowing for two reasons: it explains why some implementations drop it, and it
is a case where a *relative* error check reports failure on a gradient that is perfectly
correct -- both numbers are zero, so their relative difference is meaningless. Use absolute
error when the true gradient may vanish. (Lecture 5's BatchNorm check had the same problem.)

**TODO 3.** Verify the claim. Add `bqkv` to `msa_forward`, derive and implement its gradient,
and check it with **absolute** error. Confirm the middle third is zero to machine precision
while the first and last thirds are not.

In [ ]:
# TODO 3 -- your code here

---
## 4. The Patch Embedding Is a Convolution

ViT cuts the image into $P\times P$ patches, flattens each, and multiplies by a learned
$E$. That is a convolution with kernel $P$ and stride $P$ -- not approximately, exactly.

In [ ]:
def patchify(imgs, grid=4, patch=2):
    """(B, 8, 8) -> (B, grid*grid, patch*patch), row-major over the patch grid."""
    B = imgs.shape[0]
    out = np.empty((B, grid * grid, patch * patch))
    idx = 0
    for i in range(grid):
        for j in range(grid):
            out[:, idx] = imgs[:, i*patch:(i+1)*patch,
                               j*patch:(j+1)*patch].reshape(B, -1)
            idx += 1
    return out


digits = load_digits()
Ximg, yall = digits.images / 16.0, digits.target
Xtr_i, Xte_i, ytr, yte = train_test_split(Ximg, yall, test_size=0.3,
                                          random_state=0, stratify=yall)
print("images", Xtr_i.shape, " patchified", patchify(Xtr_i).shape)

In [ ]:
# route A: patchify, then one matmul
r = np.random.default_rng(0)
P_, D_ = 2, 6
E = r.standard_normal((P_ * P_, D_))
img = Xte_i[:4]
outA = patchify(img) @ E                       # (4, 16, D)

# route B: slide a P x P kernel with stride P
outB = np.empty((4, 4, 4, D_))
for i in range(4):
    for j in range(4):
        win = img[:, i*P_:(i+1)*P_, j*P_:(j+1)*P_].reshape(4, -1)
        outB[:, i, j] = win @ E
outB = outB.reshape(4, 16, D_)

print("max difference between the two routes:", np.abs(outA - outB).max())
print("-> the patch embedding IS a strided convolution")

Two consequences worth drawing out:

- Every ViT contains exactly one convolution, and it is the layer that injects the only
  2D structure the model ever receives.
- `nn.Conv2d(3, D, kernel_size=16, stride=16)` is how `timm` actually implements it. Not a
  trick -- the same operation, written the faster way.

---
## 5. Permutation Equivariance

$$\mathrm{Attention}(PX) = P\,\mathrm{Attention}(X)$$

Shuffle the tokens and the output is shuffled the same way -- nothing else changes.
Attention has no idea where anything is.

In [ ]:
r = np.random.default_rng(2)
X1 = r.standard_normal((1, 16, 12))
perm = r.permutation(16)

out_orig, _ = msa_forward(X1, Wqkv, Wo, h)
out_perm, _ = msa_forward(X1[:, perm], Wqkv, Wo, h)

print("max |Attn(PX) - P Attn(X)| =", np.abs(out_perm[0] - out_orig[0][perm]).max())
print("(zero up to floating point -- the layer is permutation-equivariant)")

In [ ]:
# adding position embeddings breaks the symmetry, which is exactly the point
pos = r.standard_normal((16, 12)) * 0.5
out_a, _ = msa_forward(X1 + pos, Wqkv, Wo, h)
out_b, _ = msa_forward(X1[:, perm] + pos, Wqkv, Wo, h)
print("with position embeddings, the same difference is",
      f"{np.abs(out_b[0] - out_a[0][perm]).max():.4f}  <- no longer zero")

**TODO 4.** A convolution is *translation*-equivariant but not *permutation*-equivariant.
Show both halves: take a small image, shift it by one pixel, and confirm a convolution's
output shifts identically; then permute the pixels and confirm the output is **not** the
same permutation of the original. Explain in one sentence why this makes convolution the
stronger prior for images.

In [ ]:
# TODO 4 -- your code here

---
## 6. A Complete Transformer Block

$$z' = \mathrm{MSA}(\mathrm{LN}(z)) + z, \qquad z_{\text{out}} = \mathrm{MLP}(\mathrm{LN}(z')) + z'$$

Note the **pre-norm** placement: LayerNorm sits inside the residual branch, so there is an
unobstructed identity path from input to output. The original 2017 transformer put it after
the residual add; every modern implementation, ViT included, moved it.

In [ ]:
def layernorm(x, g, b, eps=1e-5):
    """Normalises each token over its own D features -- no batch dependence."""
    m = x.mean(-1, keepdims=True)
    v = x.var(-1, keepdims=True)
    xh = (x - m) / np.sqrt(v + eps)
    return xh * g + b


def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x ** 3)))


def block_forward(z, p, h):
    zn = layernorm(z, p["g1"], p["b1"])
    z1 = msa_forward(zn, p["Wqkv"], p["Wo"], h)[0] + z        # residual 1
    zn2 = layernorm(z1, p["g2"], p["b2"])
    z2 = gelu(zn2 @ p["W1"] + p["c1"]) @ p["W2"] + p["c2"] + z1   # residual 2
    return z2


D, h = 12, 3
p = dict(g1=np.ones(D), b1=np.zeros(D), g2=np.ones(D), b2=np.zeros(D),
         Wqkv=Wqkv, Wo=Wo,
         W1=r.standard_normal((D, 4*D)) * 0.1, c1=np.zeros(4*D),
         W2=r.standard_normal((4*D, D)) * 0.1, c2=np.zeros(D))
z_in = r.standard_normal((2, 16, D))
z_out = block_forward(z_in, p, h)
print("block preserves shape:", z_in.shape, "->", z_out.shape)
print("LayerNorm output stats per token: mean %.2e, std %.4f"
      % (layernorm(z_in, p["g1"], p["b1"]).mean(-1).mean(),
         layernorm(z_in, p["g1"], p["b1"]).std(-1).mean()))

**Why LayerNorm and not BatchNorm?** LayerNorm normalises each token over its own features,
so it has no batch dependence at all: no running statistics, no train/test discrepancy, and
no degradation at batch size 1. Every failure mode from Lecture 5's `model.eval()` section
simply does not exist here. That is why transformers use it.

**Parameter count.** Attention costs $4D^2$ ($W_Q, W_K, W_V, W_O$) and the MLP costs $8D^2$
($D\to 4D\to D$), so a block is $12D^2$. Two thirds of a transformer lives in its MLPs.

In [ ]:
for name, Dv, L in [("ViT-Base", 768, 12), ("ViT-Large", 1024, 24), ("ViT-Huge", 1280, 32)]:
    attn = 4 * Dv**2 * L
    mlp = 8 * Dv**2 * L
    print(f"{name:10s} D={Dv:5d} L={L:3d}  attention {attn/1e6:6.1f}M  "
          f"MLP {mlp/1e6:6.1f}M  total {(attn+mlp)/1e6:6.1f}M")

---
## 7. Train a Real ViT, and Read Its Attention

`tiny_vit.py` (shipped alongside this notebook) assembles the pieces above into a working
Vision Transformer: patch embedding, [class] token, learned position embeddings, pre-norm
blocks, and a full backward pass. Every one of its gradients is checked against finite
differences -- the worst absolute error anywhere in the model is $2.6\times10^{-6}$.

We train it on 8x8 digits: 2x2 patches, 16 tokens, $D = 32$, 4 heads, 2 layers.

In [ ]:
import tiny_vit as tv

Ptr, Pte = patchify(Xtr_i), patchify(Xte_i)
mu = Ptr.mean((0, 1))
Ptr, Pte = Ptr - mu, Pte - mu

model = tv.TinyViT(seed=0)
hist = model.train(Ptr, ytr, iters=1500, lr=3e-3)
print("test accuracy:", round(float((model.predict(Pte) == yte).mean()), 4))

In [ ]:
# what does the [class] token look at?
pick = [np.where(yte == c)[0][0] for c in (0, 3, 4, 8)]
S, attns, _ = model.forward(Pte[pick], keep_attn=True)

fig, axes = plt.subplots(3, 4, figsize=(7, 5))
for col, idx in enumerate(pick):
    axes[0, col].imshow(Xte_i[idx], cmap="gray")
    axes[0, col].set_title(f"digit {yte[idx]} (pred {S[col].argmax()})", fontsize=9)
    for row, A in enumerate(attns):
        a = A[col, :, 0, 1:].mean(0)          # CLS row, averaged over heads
        axes[row+1, col].imshow(a.reshape(4, 4), cmap="magma")
        if col == 0:
            axes[row+1, col].set_ylabel(f"layer {row+1}")
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("where the [class] token attends")
plt.tight_layout(); plt.show()

### The position embeddings, and the pressure they need

ViT's paper shows that learned position embeddings recover 2D image topology: nearby
patches end up with similar embeddings, with row and column structure visible. Reproduce
it -- but notice what it takes.

In [ ]:
def pos_structure(m, label):
    E = m.pos[1:]
    En = E / np.linalg.norm(E, axis=1, keepdims=True)
    Sim = En @ En.T
    coords = np.array([[k // 4, k % 4] for k in range(16)], float)
    dist = np.linalg.norm(coords[:, None] - coords[None], axis=-1)
    off = ~np.eye(16, dtype=bool)
    near, far = Sim[off & (dist <= 1.5)].mean(), Sim[off & (dist >= 3.0)].mean()
    corr = np.corrcoef(dist[off], Sim[off])[0, 1]
    print(f"{label:28s} near {near:+.3f}  far {far:+.3f}  corr(dist,sim) {corr:+.3f}")
    return Sim


Sim_plain = pos_structure(model, "trained as-is:")

In [ ]:
# now train with random translations, and look again
def shift(imgs, r, mx=2):
    out = np.zeros_like(imgs)
    for i, im in enumerate(imgs):
        dy, dx = r.integers(-mx, mx + 1, 2)
        out[i] = np.roll(np.roll(im, dy, 0), dx, 1)
    return out


r2 = np.random.default_rng(0)
Xa = np.concatenate([Xtr_i, shift(Xtr_i, r2), shift(Xtr_i, r2)])
ya = np.concatenate([ytr] * 3)
model_aug = tv.TinyViT(seed=0)
model_aug.train(patchify(Xa) - mu, ya, iters=1500, lr=3e-3)
Sim_aug = pos_structure(model_aug, "trained with translations:")

fig, axes = plt.subplots(4, 4, figsize=(4.5, 4.6))
for k in range(16):
    ax = axes[k // 4, k % 4]
    ax.imshow(Sim_aug[k].reshape(4, 4), cmap="viridis", vmin=-1, vmax=1)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("cosine similarity of learned position embeddings")
plt.tight_layout(); plt.show()

Read the two lines of output together. Trained as-is, the correlation between grid distance
and embedding similarity is **positive** -- that is, nothing: every digit sits in the same
place, so a position embedding only has to be a distinguishable name tag and there is no
reason to encode geometry.

Add translations and the correlation goes clearly **negative**: nearby patches now have
similar embeddings, because the same content appears at different token indices and the
model must relate them.

The structure in ViT's paper is not automatic. It is a response to pressure -- at their
scale, supplied by 300M varied images rather than by `np.roll`.

**TODO 5.** The model has 4 heads per layer. Do they attend to different things? Plot the
[class] attention for each head separately instead of averaging, and compute the pairwise
correlation between heads. Then compare against the paper's finding that in a real ViT some
heads are local and some global from layer 1 onwards -- and consider why 16 tokens might be
too few to show it.

In [ ]:
# TODO 5 -- your code here

---
## 8. The Data-Efficiency Crossover

The lecture's central claim: the convolutional inductive bias is a *substitute for data*.
Test it directly -- same data, same budget, ViT against a small CNN, across training set
sizes.

This cell takes a few minutes.

In [ ]:
from make_figures_l6 import _train_small_cnn      # 3x3 conv -> ReLU -> pool -> linear

sizes = [100, 200, 400, 800, 1257]
print(f"{'n':>6} {'ViT':>8} {'CNN':>8} {'CNN advantage':>15}")
vit_acc, cnn_acc = [], []
for n in sizes:
    va, ca = [], []
    for s in (0, 1):
        idx = np.random.default_rng(100 + s).choice(len(Ptr), n, replace=False)
        m = tv.TinyViT(seed=s)
        m.train(Ptr[idx], ytr[idx], iters=500, lr=3e-3, seed=s)
        va.append((m.predict(Pte) == yte).mean())
        ca.append(_train_small_cnn(Xtr_i[idx], ytr[idx], Xte_i, yte, iters=500, seed=s))
    vit_acc.append(np.mean(va)); cnn_acc.append(np.mean(ca))
    print(f"{n:6d} {vit_acc[-1]:8.4f} {cnn_acc[-1]:8.4f} "
          f"{(cnn_acc[-1]-vit_acc[-1])*100:+14.1f}")

plt.figure(figsize=(5, 3))
plt.semilogx(sizes, cnn_acc, "o-", color=ACCENT, label="small CNN")
plt.semilogx(sizes, vit_acc, "o-", color=RED, label="ViT")
plt.xlabel("training images"); plt.ylabel("test accuracy")
plt.legend(); plt.grid(alpha=.3, which="both"); plt.tight_layout(); plt.show()

The CNN's advantage should fall from roughly 22 points at 100 images to about zero at 1257.
It does not merely shrink -- it **vanishes**, and the ViT overtakes.

That is ViT's thesis in miniature. At Google's scale the same curve keeps going: with
ImageNet-1k alone ViT-Large underperforms ViT-Base; with ImageNet-21k they are comparable;
with JFT-300M the bigger model wins and beats the best ResNets. *Large scale training trumps
inductive bias* -- but only at large scale, which is why your project should almost
certainly fine-tune rather than train from scratch.

---
## 9. The Same Thing in PyTorch

Everything above in the library you will actually use. **This section needs `torch`** --
if it is not installed, the cells below will fail and that is fine; read them anyway.

In [ ]:
import torch
import torch.nn as nn

class Block(nn.Module):
    def __init__(self, D, h):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(D), nn.LayerNorm(D)
        self.attn = nn.MultiheadAttention(D, h, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(D, 4*D), nn.GELU(), nn.Linear(4*D, D))

    def forward(self, x):
        y = self.n1(x)
        x = x + self.attn(y, y, y, need_weights=False)[0]     # pre-norm
        return x + self.mlp(self.n2(x))


class ViT(nn.Module):
    def __init__(self, img=8, patch=2, D=32, h=4, L=2, n_cls=10):
        super().__init__()
        n = (img // patch) ** 2
        self.embed = nn.Conv2d(1, D, patch, patch)   # the patch embedding, as a conv
        self.cls = nn.Parameter(torch.zeros(1, 1, D))
        self.pos = nn.Parameter(torch.zeros(1, n + 1, D))
        self.blocks = nn.ModuleList([Block(D, h) for _ in range(L)])
        self.norm, self.head = nn.LayerNorm(D), nn.Linear(D, n_cls)
        nn.init.trunc_normal_(self.pos, std=0.02)

    def forward(self, x):
        z = self.embed(x).flatten(2).transpose(1, 2)          # (B, n, D)
        z = torch.cat([self.cls.expand(len(z), -1, -1), z], 1) + self.pos
        for b in self.blocks:
            z = b(z)
        return self.head(self.norm(z)[:, 0])                  # the [class] token


net = ViT()
print(net)
print("parameters:", sum(p.numel() for p in net.parameters()))

In [ ]:
# train it
Xt = torch.tensor(Xtr_i, dtype=torch.float32)[:, None]
yt = torch.tensor(ytr)
Xv = torch.tensor(Xte_i, dtype=torch.float32)[:, None]
yv = torch.tensor(yte)

opt = torch.optim.AdamW(net.parameters(), lr=3e-3, weight_decay=0.05)
lossf = nn.CrossEntropyLoss()
for epoch in range(30):
    perm = torch.randperm(len(Xt))
    for i in range(0, len(Xt), 128):
        j = perm[i:i+128]
        opt.zero_grad()
        lossf(net(Xt[j]), yt[j]).backward()
        opt.step()
    if epoch % 10 == 9:
        with torch.no_grad():
            acc = (net(Xv).argmax(1) == yv).float().mean().item()
        print(f"epoch {epoch+1:3d}  test accuracy {acc:.4f}")

Compare the `ViT.forward` above against the paper's four equations. They line up term for
term: `embed` is $x_pE$, `cat` prepends $x_{\text{class}}$, `+ self.pos` is $E_{\text{pos}}$,
each `Block` is one MSA/MLP pair, and `[:, 0]` selects the [class] token for the head.

`timm`'s production implementation is the same thing with more options.

---
## Exercises

**1. Attention entropy through depth.** Measure the entropy of the attention distributions
in layer 1 versus layer 2 of the trained model. Do later layers attend more sharply or more
broadly? Compare with the entropy of a uniform distribution over 17 tokens, $\ln 17$.

**2. Kill the position embeddings.** Set `model.pos = 0` after training and re-measure
accuracy. Then retrain from scratch with position embeddings permanently zero. The two
numbers tell you different things -- explain what each one measures.

**3. Window attention.** Implement Swin's W-MSA: restrict attention to non-overlapping
$2\times2$ windows of the $4\times4$ patch grid. Verify the cost formula
$\Omega = 4hwC^2 + 2M^2hwC$ against an operation count, then train and compare accuracy.
Does the locality prior help at this data scale? (Predict before you run it.)

**4. Shifted windows.** Extend exercise 3 so that alternating blocks shift the window
partition by one patch. Confirm that without the shift, patches in different windows never
influence each other -- you can show this by perturbing one patch and measuring which
outputs change.

**5. Distillation, in miniature.** Train the small CNN, then train a ViT on its soft outputs
instead of the hard labels (DeiT's idea without the extra token). At $n = 200$, does the ViT
beat the ViT trained on labels alone? This is the "learn the inductive bias from a convnet
teacher" claim, at a scale you can run in a minute.

**6. Patch size.** Retrain with $1\times1$ patches (64 tokens) and $4\times4$ patches
(4 tokens). Plot accuracy and training time against patch size, and explain the trade-off
in terms of sequence length and how much structure the embedding throws away.